# Energy Simulation with PyBaMM

This notebook demonstrates how to perform energy/capacity simulation using PyBaMM with BYD Blade Prismatic 135Ah cell parameters.

## Overview
- **Cell**: BYD Blade Prismatic 135Ah LFP
- **Chemistry**: LFP (Lithium Iron Phosphate)
- **Form Factor**: Prismatic
- **Nominal Voltage**: 3.3V
- **Capacity**: 135Ah
- **Simulations**: Full discharge energy characterization at multiple C-rates

In [ ]:
# Import required libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pybamm
from pathlib import Path
from model_library import simulate_energy

## 1. Load Cell Parameters

Load the cell design from the manifest file.

In [ ]:
# Load cell manifest
manifest_path = Path("../cells/BYD_Blade_Prismatic_135Ah_manifest.json")

with open(manifest_path, 'r') as f:
    cell_design_manifest = json.load(f)

# ============================================================================
# OPERATING CONDITIONS FOR ENERGY TEST
# ============================================================================
energy_config = {
    "temperature_K": 298.15,  # 25°C
    "initial_soc": 1.0,  # 100% SOC (full charge)
    "c_rates": [0.2, 0.5, 1.0, 2.0],  # Multiple C-rates for energy sweep
    "lower_voltage_cutoff": 2.5,  # Lower voltage cutoff [V]
}

## 2. Run Energy Simulation

Run the energy simulation using the SPMe model with automatic capacity calibration at multiple C-rates.

In [ ]:
energy_results = simulate_energy(cell_design_manifest=cell_design_manifest, simulation_config=energy_config)

## 3. Visualize Results

In [ ]:
# Filter successful results
successful_results = [r for r in energy_results if r['success']]

if len(successful_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Plot 1: Voltage vs Time for all C-rates
    ax1 = axes[0, 0]
    for r in successful_results:
        time_min = (r['time_s'] - r['time_s'][0]) / 60
        ax1.plot(time_min, r['voltage_V'], linewidth=2, label=f"{r['c_rate']}C")
    ax1.set_xlabel('Time [min]')
    ax1.set_ylabel('Voltage [V]')
    ax1.set_title('Voltage vs Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Voltage vs Capacity for all C-rates
    ax2 = axes[0, 1]
    for r in successful_results:
        capacity = r['capacity_Ah'] - r['capacity_Ah'][0]
        ax2.plot(capacity, r['voltage_V'], linewidth=2, label=f"{r['c_rate']}C")
    ax2.set_xlabel('Discharge Capacity [Ah]')
    ax2.set_ylabel('Voltage [V]')
    ax2.set_title('Voltage vs Capacity')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Plot 3: Energy vs C-rate (bar chart)
    ax3 = axes[1, 0]
    c_rates = [r['c_rate'] for r in successful_results]
    energies = [r['discharge_energy_Wh'] for r in successful_results]
    bars = ax3.bar([f"{c}C" for c in c_rates], energies, color='steelblue', alpha=0.8)
    ax3.set_xlabel('C-rate')
    ax3.set_ylabel('Energy [Wh]')
    ax3.set_title('Discharge Energy vs C-rate')
    ax3.grid(True, alpha=0.3, axis='y')
    # Add value labels on bars
    for bar, energy in zip(bars, energies):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                 f'{energy:.1f}', ha='center', va='bottom', fontsize=10)

    # Plot 4: Temperature vs Time for all C-rates
    ax4 = axes[1, 1]
    for r in successful_results:
        time_min = (r['time_s'] - r['time_s'][0]) / 60
        temp_C = r['temperature_K'] - 273.15
        ax4.plot(time_min, temp_C, linewidth=2, label=f"{r['c_rate']}C")
    ax4.set_xlabel('Time [min]')
    ax4.set_ylabel('Temperature [°C]')
    ax4.set_title('Temperature vs Time')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.suptitle(f'BYD Blade 135Ah - Energy Simulation ({energy_config["temperature_K"]-273.15:.0f}°C)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No successful simulations to plot.")

In [ ]:
# Create summary DataFrame
if len(successful_results) > 0:
    summary_data = []
    for r in successful_results:
        summary_data.append({
            'C-rate': f"{r['c_rate']}C",
            'Capacity [Ah]': r['discharge_capacity_Ah'],
            'Energy [Wh]': r['discharge_energy_Wh'],
            'Avg Voltage [V]': r['average_voltage_V'],
            'Duration [min]': r['duration_min'],
            'Max Temp [°C]': r['max_temperature_C'],
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\nEnergy Simulation Results:")
    print(summary_df.to_string(index=False))